cd "D:\vscode文件夹\大模型学习笔记"
git add .
git commit -m "更新xxx笔记并添加图解"
git push

![](images/2026-11-04-15-13-56.png)
![](images/2026-11-04-15-14-28.png)
![](images/2026-11-04-15-14-44.png)
![](images/2026-11-04-15-15-13.png)
![](images/2026-11-04-15-15-30.png)
![](images/2026-11-04-15-16-16.png)
![](images/2026-11-04-15-16-46.png)

### WordPiece
##### WordPiece是谷歌提出并在 BERT 中采用的经典子词分词算法
![](images/2026-11-04-15-19-05.png)
![](images/2026-11-04-15-19-27.png)
![](images/2026-11-04-15-19-47.png)
![](images/2026-11-04-15-20-22.png)
![](images/2026-11-04-15-20-48.png)
![](images/2026-11-04-15-21-39.png)
![](images/2026-11-04-15-21-59.png)
![](images/2026-11-04-15-22-20.png)
![](images/2026-11-04-15-23-06.png)
![](images/2026-11-04-15-23-15.png)

### Word2Vec
![](images/2026-11-04-15-46-41.png)
![](images/2026-11-04-15-47-06.png)
![](images/2026-11-04-15-52-44.png)
![](images/2026-11-04-15-52-53.png)
![](images/2026-11-04-15-53-01.png)
![](images/2026-11-04-15-53-17.png)
![](images/2026-11-04-15-56-49.png)
![](images/2026-11-04-15-57-04.png)
![](images/2026-11-04-15-57-12.png)
![](images/2026-11-04-15-57-32.png)
![](images/2026-11-04-15-57-43.png)

### Attention
![](images/2026-11-04-17-06-39.png)
![](images/2026-11-04-17-08-05.png)
![](images/2026-11-04-17-13-58.png)
![](images/2026-11-04-17-14-11.png)
![](images/2026-11-04-17-14-20.png)
![](images/2026-11-04-17-14-38.png)
![](images/2026-11-04-17-15-22.png)

##### 复杂度 O(N^2)
![](images/2026-11-04-19-31-16.png)
![](images/2026-11-04-19-31-23.png)
![](images/2026-11-04-19-31-35.png)
![](images/2026-11-04-19-31-42.png)

##### KV Cache（键值缓存）
![](images/2026-11-04-20-40-41.png)
![](images/2026-11-04-20-41-08.png)
![](images/2026-11-04-20-41-45.png)
![](images/2026-11-04-20-42-11.png)
![](images/2026-11-04-20-42-22.png)
![](images/2026-11-04-20-42-33.png)
![](images/2026-11-04-20-46-57.png)
![](images/2026-11-04-20-47-15.png)
![](images/2026-11-04-20-47-29.png)
![](images/2026-11-04-20-48-24.png)
![](images/2026-11-04-20-49-06.png)
![](images/2026-11-04-20-49-16.png)
![](images/2026-11-04-20-49-26.png)
![](images/2026-11-04-20-50-06.png)
![](images/2026-11-04-20-52-24.png)
![](images/2026-11-04-20-57-27.png)
![](images/2026-11-04-21-31-57.png) 

## MHA(Multi-head Attention)

In [ ]:
import torch
from torch import nn   #import torch.nn as nn 
import math

class MutiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        ## 初始化Q、K、V投影矩阵
        self.q_linear = nn.Linear(hidden_size, hidden_size,bias = False)
        self.k_linear = nn.Linear(hidden_size, hidden_size,bias = False)
        self.v_linear = nn.Linear(hidden_size, hidden_size,bias = False)
        ## 输出线性层
        self.o_linear = nn.Linear(hidden_size, hidden_size,bias = False)
        
    def forward(self, hidden_state, attention_mask=None):
        batch_size , seq_len , _ = hidden_state.shape
        ## 形状变换: (batch_size, seq_len, hidden_size) -> (batch_size, num_heads, seq_len, head_dim)
        query = self.q_linear(hidden_state).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        key = self.k_linear(hidden_state).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        value = self.v_linear(hidden_state).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        #query = self.q_linear(hidden_state).reshape(batch_size,-1,self.num_heads, self.head_dim).transpose(1,2)也行

        ## 计算注意力分数
        #attention_scores = torch.matmul(query, key.transpose(-1, -2))/math.sqrt(self.head_dim)
        attention_scores = (query @ key.transpose(-1, -2))/math.sqrt(self.head_dim) #*self.head_dim**-0.5也可以

        if attention_mask != None:
            attention_scores = attention_scores.masked_fill(attention_mask == 0,float('-inf'))
        ## 对注意力分数进行归一化
        attention_probs = torch.softmax(attention_scores, dim=-1)
        output = attention_probs @ value
        #output = torch.matmul(attention_probs, value)
        ## 对注意力输出进行拼接
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.head_dim * self.num_heads)
        #output = output.transpose(1, 2).reshape(batch_size, -1, self.head_dim * self.num_heads)也行

        output = self.o_linear(output)
        return output



![](images/2026-11-04-23-13-57.png)
![](images/2026-11-04-23-14-30.png)
![](images/2026-11-04-23-14-41.png)
![](images/2026-11-04-23-15-16.png)

![](images/2026-11-04-23-55-49.png)
![](images/2026-11-04-23-56-25.png)

###### 填充掩码、因果掩码
![](images/2026-11-05-09-03-27.png)
![](images/2026-11-05-09-05-35.png)
![](images/2026-11-05-09-05-59.png)
![](images/2026-11-05-09-22-09.png)
![](images/2026-11-05-10-11-59.png)
![](images/2026-11-05-10-14-49.png)
![](images/2026-11-05-10-15-05.png)
![](images/2026-11-05-10-15-14.png)
![](images/2026-11-05-10-15-27.png)
![](images/2026-11-05-10-15-57.png)
![](images/2026-11-05-10-16-08.png)

```python
output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.head_dim * self.num_heads)
```
![](images/2026-11-05-21-29-01.png)
![](images/2026-11-05-11-21-42.png)
![](images/2026-11-05-11-21-57.png)
![](images/2026-11-05-21-27-32.png)
![](images/2026-11-05-21-29-17.png)
![](images/2026-11-05-21-29-35.png)
![](images/2026-11-05-21-31-01.png)

##### 为什么前面不需要contingous后面就需要呢？区分view发生在转置前还是转置后!!!
![](images/2026-11-05-21-34-47.png)
![](images/2026-11-05-21-35-00.png)
![](images/2026-11-05-21-35-54.png)

In [ ]:
import torch
from torch import nn   #import torch.nn as nn 
import math

class MutiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        ## QKV 投影矩阵合并
        self.qkv_proj = nn.Linear(hidden_size, 3 * hidden_size, bias=False)
        ## 输出线性层
        self.o_linear = nn.Linear(hidden_size, hidden_size,bias = False)
        
    def forward(self, hidden_state, attention_mask=None):
        batch_size , seq_len , _ = hidden_state.shape
        
        #单次投影生成 Q, K, V，随后切分为三个张量
        qkv = self.qkv_proj(hidden_state)  # (batch_size, seq_len, 3 * hidden_size)
        query, key, value = qkv.chunk(3, dim=-1)    

        ## 形状变换: (batch_size, seq_len, hidden_size) -> (batch_size, num_heads, seq_len, head_dim)
        query = self.q_linear(hidden_state).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        key = self.k_linear(hidden_state).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        value = self.v_linear(hidden_state.)view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        #query = self.q_linear(hidden_state).reshape(batch_size,-1,self.num_heads, self.head_dim).transpose(1,2)也行

        ## 计算注意力分数
        #attention_scores = torch.matmul(query, key.transpose(-1, -2))/math.sqrt(self.head_dim)
        attention_scores = (query @ key.transpose(-1, -2))/math.sqrt(self.head_dim) #*self.head_dim**-0.5也可以

        if attention_mask != None:
            attention_scores = attention_scores.masked_fill(attention_mask == 0,float('-inf'))
        ## 对注意力分数进行归一化
        attention_probs = torch.softmax(attention_scores, dim=-1)
        output = attention_probs @ value
        #output = torch.matmul(attention_probs, value)
        ## 对注意力输出进行拼接
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.head_dim * self.num_heads)
        #output = output.transpose(1, 2).reshape(batch_size, -1, self.head_dim * self.num_heads)也行

        output = self.o_linear(output)
        return output



##### QKV 投影矩阵合并
![](images/2026-11-05-21-45-33.png)
![](images/2026-11-05-21-46-11.png)
![](images/2026-11-05-21-46-36.png)
![](images/2026-11-05-21-51-00.png)
![](images/2026-11-05-21-49-55.png)
![](images/2026-11-05-21-51-48.png)

## MQA(Multi-query Attention)

In [ ]:
import math
import torch
from torch import nn

class MultiQueryAttention(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        # 预先计算缩放因子，避免在 forward 循环中重复计算或创建小张量
        self.scale = self.head_dim ** -0.5
        
        # MQA 核心结构：Q 维持多头总维度，K 和 V 仅保留 1 个头（维度为 head_dim）
        # 显式关闭 bias，符合现代主流大模型规范
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, self.head_dim, bias=False)
        self.v_proj = nn.Linear(hidden_size, self.head_dim, bias=False)
        
        # 输出投影层
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        
    def forward(self, hidden_state: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        batch_size, seq_len, _ = hidden_state.shape
        
        # 1. 线性投影
        # query 形状: (batch_size, seq_len, hidden_size)
        # key / value 形状: (batch_size, seq_len, head_dim)
        query = self.q_proj(hidden_state)
        key = self.k_proj(hidden_state)
        value = self.v_proj(hidden_state)
        
        # 2. 形状调整与头切分
        # Q 保持多头形状: (batch_size, num_heads, seq_len, head_dim)
        query = query.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        # K 与 V 仅有 1 个头，显式插入头维度: (batch_size, 1, seq_len, head_dim)
        key = key.view(batch_size, seq_len, 1, self.head_dim).transpose(1, 2)
        value = value.view(batch_size, seq_len, 1, self.head_dim).transpose(1, 2)
        
        # 3. 计算注意力分数
        # key.transpose(-1, -2) 形状为: (batch_size, 1, head_dim, seq_len)
        # PyTorch 的广播机制会自动将 K 的头维度 1 广播到与 Q 的 num_heads 匹配
        # attention_scores 形状: (batch_size, num_heads, seq_len, seq_len)
        attention_scores = torch.matmul(query, key.transpose(-1, -2)) * self.scale
        
        # 4. 掩码处理（使用当前精度的物理极小值，避免数值溢出）
        if attention_mask is not None:
            attention_scores = attention_scores.masked_fill(attention_mask == 0, float('-inf'))
        # 5. Softmax 与加权求和
        attention_probs = torch.softmax(attention_scores, dim=-1)
        # value 形状为 (batch_size, 1, seq_len, head_dim)，此处再次自动广播头维度
        # output 形状: (batch_size, num_heads, seq_len, head_dim)
        output = torch.matmul(attention_probs, value)
        
        # 6. 多头还原与拼接
        # 形状变换: (batch_size, num_heads, seq_len, head_dim) -> (batch_size, seq_len, hidden_size)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        # 7. 最终线性投影
        return self.o_proj(output)

![](images/2026-11-05-22-27-07.png)
![](images/2026-11-05-22-28-17.png)
![](images/2026-11-05-22-36-01.png)
![](images/2026-11-05-22-36-13.png)

## GQA(Grouped-query Attention)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GroupQueryAttention(nn.Module):
    def __init__(self, embed_dim: int = 512, num_heads: int = 8, groups: int = 2, dropout: float = 0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.groups = groups                      # KV 的头数（组数）
        self.head_dim = embed_dim // num_heads    # 每个头的维度
        self.group_heads = num_heads // groups    # 每一组里有几个 Q 头共享同一个 KV 头
        
        # 预先计算缩放系数（乘倒数代替除法，速度更快）
        self.scale = self.head_dim ** -0.5
        
        # 现代大模型规范：显式加上 bias=False
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        # 注意：K 和 V 只需要投影出 groups 个头即可
        self.k_proj = nn.Linear(embed_dim, self.groups * self.head_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, self.groups * self.head_dim, bias=False)
        
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor, key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        
        # 1. 线性投影
        q = self.q_proj(x)  # (batch, seq_len, embed_dim)
        k = self.k_proj(x)  # (batch, seq_len, groups * head_dim)
        v = self.v_proj(x)  # (batch, seq_len, groups * head_dim)
        
        # 2. 统一调整多头形状（保持 Q、K、V 风格完全对称统一）
        # 形状变为: (batch_size, 头数, seq_len, head_dim)
        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.groups, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.groups, self.head_dim).transpose(1, 2)
        
        # 3. 核心步骤：将每个组的 1 个 KV 头，复制扩充为 group_heads 个头以匹配 Q
        # 使用 repeat_interleave 替代冗长的 unsqueeze + expand + view
        # 形状变换: (batch, groups, seq_len, head_dim) -> (batch, num_heads, seq_len, head_dim)
        k = k.repeat_interleave(self.group_heads, dim=1)
        v = v.repeat_interleave(self.group_heads, dim=1)
        
        # 4. 计算注意力分数
        # (batch, num_heads, seq_len, head_dim) @ (batch, num_heads, head_dim, seq_len)
        attn_scores = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        
        # 5. 掩码填充
        if key_padding_mask is not None:
            attn_scores = attn_scores.masked_fill(key_padding_mask == 0, float('-inf'))
            
        # 6. Softmax 归一化与 Dropout
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # 7. 加权求和并恢复原始序列维度
        output = torch.matmul(attn_weights, v)  # (batch, num_heads, seq_len, head_dim)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        return self.out_proj(output)

![](images/2026-11-06-09-24-32.png)
![](images/2026-11-06-09-24-47.png)
![](images/2026-11-06-09-25-39.png)
![](images/2026-11-06-09-27-42.png)
![](images/2026-11-06-09-28-15.png)
![](images/2026-11-06-09-29-32.png)
![](images/2026-11-06-09-29-48.png)

```python
k = k.repeat_interleave(self.group_heads, dim=1)
v = v.repeat_interleave(self.group_heads, dim=1)
```
![](images/2026-11-06-09-30-12.png)
![](images/2026-11-06-09-31-02.png)
![](images/2026-11-06-09-31-11.png)
![](images/2026-11-06-09-31-34.png)
![](images/2026-11-06-09-32-05.png)
![](images/2026-11-06-09-32-14.png)
![](images/2026-11-06-09-32-27.png)
![](images/2026-11-06-09-32-37.png)

### Dropout
![](images/2026-11-06-10-10-56.png)
![](images/2026-11-06-10-11-05.png)

![](images/2026-11-06-10-20-49.png)
![](images/2026-11-06-10-21-23.png)
![](images/2026-11-06-10-23-40.png)
![](images/2026-11-06-10-24-22.png)
![](images/2026-11-06-10-24-33.png)
![](images/2026-11-06-10-24-41.png)

### 残差连接
![](images/2026-11-06-17-11-09.png)
![](images/2026-11-06-17-11-23.png)
![](images/2026-11-06-17-12-11.png)
![](images/2026-11-06-17-12-22.png)
![](images/2026-11-06-17-12-34.png)
![](images/2026-11-06-17-13-03.png)
![](images/2026-11-06-18-52-01.png)
![](images/2026-11-06-18-52-18.png)
![](images/2026-11-06-18-52-48.png)
![](images/2026-11-06-18-53-13.png)
![](images/2026-11-06-18-58-42.png)
![](images/2026-11-06-18-58-56.png)

### Layer Norm
![](images/2026-11-06-20-37-39.png)
![](images/2026-11-06-20-37-47.png)
![](images/2026-11-06-20-37-54.png)
![](images/2026-11-06-20-38-00.png)
![](images/2026-11-06-20-39-03.png)

### pre-Norm & post-Norm
![](images/2026-11-06-22-48-51.png)
![](images/2026-11-06-22-49-05.png)
![](images/2026-11-06-22-49-45.png)
![](images/2026-11-06-22-49-53.png)
![](images/2026-11-06-22-50-28.png)
![](images/2026-11-06-22-50-44.png)
![](images/2026-11-06-22-51-00.png)
![](images/2026-11-06-22-51-41.png)
![](images/2026-11-06-22-52-20.png)

### 手撕LayerNorm

In [ ]:
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-5):
        """
        参数:
            normalized_shape: 需要归一化的特征维度大小 (例如 hidden_size)
            eps: 防止除零的微小常数
        """
        super().__init__()
        self.normalized_shape = normalized_shape
        self.eps = eps
        
        # 两个可学习参数：缩放权重 gamma 与 偏置 beta
        # gamma 初始化全为 1，beta 初始化全为 0
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 输入 x 形状通常为: (batch_size, seq_len, hidden_size)
        
        # 1. 计算最后一个维度 (特征维度) 的均值
        # keepdim=True 极其关键：保持维度为 (batch, seq_len, 1)，便于后续广播减法
        mean = x.mean(dim=-1, keepdim=True)
        
        # 2. 计算方差
        # unbiased=False: 深度学习中使用总体方差 (除以 N)，而非样本无偏方差 (除以 N-1)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        
        # 3. 标准化：减均值，除以标准差 (加 eps 防止分母为 0)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        
        # 4. 仿射变换：逐元素乘以 gamma 并加上偏置 beta (自动通过广播机制扩展)
        out = x_norm * self.gamma + self.beta
        
        return out

![](images/2026-11-06-23-32-11.png)
![](images/2026-11-06-23-47-37.png)
![](images/2026-11-06-23-48-00.png)
![](images/2026-11-06-23-54-37.png)
![](images/2026-11-07-00-04-47.png)
![](images/2026-11-07-10-13-37.png)
![](images/2026-11-07-10-14-33.png)
![](images/2026-11-07-10-15-04.png)
![](images/2026-11-07-10-15-22.png)

![](images/2026-11-07-10-47-57.png)
![](images/2026-11-07-10-48-09.png)
![](images/2026-11-07-10-48-23.png)
![](images/2026-11-07-10-48-34.png)
![](images/2026-11-07-10-48-56.png)
![](images/2026-11-07-10-49-06.png)
![](images/2026-11-07-10-56-34.png)
![](images/2026-11-07-11-01-23.png)
![](images/2026-11-07-11-02-54.png)
它们函数导入的方式不一样，nn.Parameter是因为已经在__init__.py文件里面导入好了，所以如果没加from torch.nn.parameter import Parameter这句话，那就可以用nn.Parameter快捷导入；如果是已经加入from torch.nn.parameter import Parameter，那么就是说从parameter这个文件吧Parameter函数导入到当前脚本，所以可以直接Parameter

![](images/2026-11-06-23-47-25.png)

### 手撕RMS Norm（Root Mean Square Layer Normalization）

In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        """
        参数:
            dim: 特征维度大小 (即 hidden_size，如 4096)
            eps: 防止除以 0 的极小常数
        """
        super().__init__()
        self.eps = eps
        self.dim = dim
        # 仅有一个可学习缩放参数 gamma (通常命名为 weight)
        # 必须用 nn.Parameter 包装，初始化全为 1
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 输入 x 形状通常为: (batch_size, seq_len, dim)
        
        # 1. 计算均方根 (RMS)：在最后一个维度求平方的均值，再开平方根
        # keepdim=True 保持形状为 (batch, seq_len, 1)，便于广播除法
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        
        # 2. 归一化并乘以可学习缩放参数
        # (batch, seq_len, dim) / (batch, seq_len, 1) * (dim,)
        return (x / rms) * self.weight

### GLU（Gated Linear Unit 门控线性单元）
![](images/2026-11-07-15-05-59.png)
在传统 Transformer（如 GPT-2、BERT）中，FFN 确实采用两层全连接加一个激活函数（如 ReLU 或 GELU）；但在现代所有主流大模型（LLaMA 1/2/3、Qwen、DeepSeek、Mistral 等）中，传统的 FFN 已经全部被替换成了 SwiGLU。
![](images/2026-11-07-15-03-56.png)
![](images/2026-11-07-15-04-10.png)
![](images/2026-11-07-15-04-33.png)
![](images/2026-11-07-15-04-55.png)
![](images/2026-11-07-15-05-43.png)
![](images/2026-11-07-15-22-45.png)
![](images/2026-11-07-15-06-22.png)
![](images/2026-11-07-15-06-35.png)
![](images/2026-11-07-15-06-48.png)
![](images/2026-11-07-15-07-07.png)
![](images/2026-11-07-15-07-32.png)

In [ ]:
### 手撕SwiGLU
import torch
import torch.nn as nn
import torch.nn.functional as F

class SwiGLU(nn.Module):
    def __init__(self, in_features: int, hidden_features: int):
        super().__init__()
        # 1. 负责阀门分支的线性层 (Gate)
        self.w_gate = nn.Linear(in_features, hidden_features, bias=False)
        # 2. 负责内容信号分支的线性层 (Up)
        self.w_up = nn.Linear(in_features, hidden_features, bias=False)
        # 3. 负责最终降维输出的线性层 (Down)
        self.w_down = nn.Linear(hidden_features, in_features, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 输入 x 形状: (batch_size, seq_len, in_features)
        
        # 步骤 1: 信号路生成与阀门路生成
        # F.silu 就是带有平滑特性的 Swish 激活函数
        gate = F.silu(self.w_gate(x))
        up = self.w_up(x)
        
        # 步骤 2: 两路逐元素相乘 (门控过滤)
        gated_hidden = gate * up
        
        # 步骤 3: 降维输出回原始维度
        return self.w_down(gated_hidden)

![](images/2026-11-07-15-27-43.png)
![](images/2026-11-07-15-27-58.png)
![](images/2026-11-07-15-31-55.png)
![](images/2026-11-07-15-32-21.png)

### 位置编码
![](images/2026-11-08-11-13-12.png)
![](images/2026-11-08-11-13-21.png)
![](images/2026-11-08-15-27-21.png)
![](images/2026-11-08-15-30-11.png)
![](images/2026-11-08-15-30-21.png)
![](images/2026-11-08-15-30-30.png)
![](images/2026-11-08-15-31-07.png)

![](images/2026-11-08-15-37-56.png)
![](images/2026-11-08-15-38-08.png)
![](images/2026-11-08-15-38-24.png)
![](images/2026-11-08-15-38-36.png)

![](images/2026-11-08-15-39-03.png)
![](images/2026-11-08-15-39-18.png)
![](images/2026-11-08-15-40-30.png)
![](images/2026-11-08-15-40-44.png)
![](images/2026-11-08-15-41-00.png)
![](images/2026-11-08-15-41-35.png)
![](images/2026-11-08-15-41-47.png)

![](images/2026-11-08-15-43-54.png)

![](images/2026-11-08-15-44-16.png)
![](images/2026-11-08-15-44-28.png)
![](images/2026-11-08-15-44-39.png)
![](images/2026-11-08-15-44-59.png)
![](images/2026-11-08-15-45-14.png)

![](images/2026-11-08-15-46-35.png)
![](images/2026-11-08-15-46-48.png)

![](images/2026-11-08-15-47-26.png)
![](images/2026-11-08-15-47-57.png)
![](images/2026-11-08-15-48-29.png)
![](images/2026-11-10-17-19-52.png)
![](images/2026-11-10-17-19-59.png)
![](images/2026-11-10-17-20-04.png)
![](images/2026-11-10-17-20-11.png)
![](images/2026-11-10-17-20-23.png)
![](images/2026-11-10-17-21-03.png)

![](images/2026-11-08-15-51-18.png)
![](images/2026-11-08-15-51-28.png)
![](images/2026-11-08-15-51-45.png)
![](images/2026-11-08-15-52-55.png)
![](images/2026-11-08-15-53-04.png)
![](images/2026-11-08-15-53-22.png)
![](images/2026-11-08-15-53-49.png)
![](images/2026-11-08-15-54-38.png)
![](images/2026-11-08-15-55-18.png)
![](images/2026-11-08-15-55-34.png)

### ROPE
https://mp.weixin.qq.com/s/-1xVXjoM0imXMC7DKqo-Gw
![](images/2026-11-10-10-47-30.png)
![](images/2026-11-10-10-47-49.png)
![](images/2026-11-10-10-48-04.png)
![](images/2026-11-08-18-47-50.png)

#### 绝对位置编码，主要包括训练式位置编码与Sinusoidal位置编码。
##### 绝对位置编码
![](images/2026-11-10-10-50-33.png)
![](images/2026-11-10-10-50-38.png)
![](images/2026-11-10-10-50-43.png)

##### Sinusoidal位置编码
![](images/2026-11-10-10-53-17.png)
![](images/2026-11-10-10-53-49.png)

![](images/2026-11-10-11-06-51.png)
![](images/2026-11-10-11-08-14.png)
![](images/2026-11-10-11-08-34.png)
![](images/2026-11-10-11-08-43.png)
![](images/2026-11-10-11-09-05.png)
![](images/2026-11-10-11-10-20.png)
![](images/2026-11-10-14-22-53.png)
![](images/2026-11-10-14-23-10.png)
![](images/2026-11-10-14-23-22.png)
![](images/2026-11-10-14-24-04.png)
![](images/2026-11-10-14-24-30.png)
![](images/2026-11-10-14-25-06.png)
![](images/2026-11-10-14-25-19.png)

![](images/2026-11-10-14-34-32.png)
![](images/2026-11-10-14-34-43.png)
![](images/2026-11-10-14-34-56.png)
![](images/2026-11-10-14-35-14.png)
![](images/2026-11-10-14-35-37.png)
![](images/2026-11-10-14-35-45.png)
![](images/2026-11-10-14-36-48.png)

![](images/2026-11-10-14-41-13.png)
![](images/2026-11-10-14-41-33.png)
![](images/2026-11-10-14-56-56.png)
![](images/2026-11-10-15-11-00.png)
![](images/2026-11-10-15-10-53.png)
![](images/2026-11-10-15-26-31.png)

![](images/2026-11-10-15-26-42.png)
![](images/2026-11-10-15-48-30.png)
![](images/2026-11-10-15-27-13.png)
![](images/2026-11-10-15-29-48.png)
![](images/2026-11-10-15-29-59.png)
![](images/2026-11-10-15-30-18.png)